In [1]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.8.4.1 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cufft-cu12==1

## Import Libraries

In [2]:
import numpy as np
import pandas as pd
import json
import torch
import time
import evaluate
from tqdm import tqdm
# pip install accelerate
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainer
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, GenerationConfig
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import Dataset, load_dataset

2025-05-10 10:14:48.188586: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746872088.425827      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746872088.500853      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# with open('/kaggle/input/stanford-question-answering-dataset/train-v1.1.json', 'rb') as f:
#     trainfile = json.load(f)

# context = {}
# title = {}
# qas = {'title_id':[], 'context_id':[], 'question':[], 'answer':[]}
# ctr1 = 0
# ctr2 = 0
# for p in trainfile['data']:
#     title[ctr1] = p['title']
    
#     for d in p['paragraphs']:    
#         context[ctr2] = d['context']

#         for qas1 in d['qas']:
#             for ans1 in qas1['answers']:
#                 qas['title_id'].append(ctr1)
#                 qas['context_id'].append(ctr2)
#                 qas['question'].append(qas1['question'])
#                 qas['answer'].append(ans1['text'])
#         ctr2+=1
#     ctr1+=1

# df_train = pd.DataFrame(qas, columns=qas.keys()).dropna().drop_duplicates().reset_index(drop=True)
# print(f"df_train.shape: {df_train.shape}")

# with open('/kaggle/input/stanford-question-answering-dataset/dev-v1.1.json', 'rb') as f:
#     testfile = json.load(f)

# # context = {}
# # title = {}
# qas = {'title_id':[], 'context_id':[], 'question':[], 'answer':[]}
# # ctr1 = 0
# # ctr2 = 0
# for p in testfile['data']:
#     title[ctr1] = p['title']
    
#     for d in p['paragraphs']:    
#         context[ctr2] = d['context']

#         for qas1 in d['qas']:
#             for ans1 in qas1['answers']:
#                 qas['title_id'].append(ctr1)
#                 qas['context_id'].append(ctr2)
#                 qas['question'].append(qas1['question'])
#                 qas['answer'].append(ans1['text'])
#         ctr2+=1
#     ctr1+=1

# df_test = pd.DataFrame(qas, columns=qas.keys()).dropna().drop_duplicates().reset_index(drop=True)
# print(f"df_test.shape: {df_test.shape}")

## Download Flan-T5 model & tokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", torch_dtype=torch.bfloat16, device_map='auto')

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(model))

trainable model parameters: 247577856
all model parameters: 247577856
percentage of trainable model parameters: 100.00%


## Load the train/val/test dataset

In [6]:
train_df = pd.read_csv('/kaggle/input/dialogsum/CSV/train.csv')
print(f"train_df.shape: {train_df.shape}")
train_df.head()

train_df.shape: (12460, 4)


,id,dialogue,summary,topic
0,train_0,"#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. ...","Mr. Smith's getting a check-up, and Doctor Haw...",get a check-up
1,train_1,"#Person1#: Hello Mrs. Parker, how have you bee...",Mrs Parker takes Ricky for his vaccines. Dr. P...,vaccines
2,train_2,"#Person1#: Excuse me, did you see a set of key...",#Person1#'s looking for a set of keys and asks...,find keys
3,train_3,#Person1#: Why didn't you tell me you had a gi...,#Person1#'s angry because #Person2# didn't tel...,have a girlfriend
4,train_4,"#Person1#: Watsup, ladies! Y'll looking'fine t...",Malik invites Nikki to dance. Nikki agrees if ...,dance


In [7]:
val_df = pd.read_csv('/kaggle/input/dialogsum/CSV/validation.csv')
print(f"val_df.shape: {val_df.shape}")
val_df.head()

val_df.shape: (500, 4)


,id,dialogue,summary,topic
0,dev_0,"#Person1#: Hello, how are you doing today?\n#P...",#Person2# has trouble breathing. The doctor as...,see a doctor
1,dev_1,#Person1#: Hey Jimmy. Let's go workout later t...,#Person1# invites Jimmy to go workout and pers...,do exercise
2,dev_2,#Person1#: I need to stop eating such unhealth...,#Person1# plans to stop eating unhealthy foods...,healthy foods
3,dev_3,#Person1#: Do you believe in UFOs?\n#Person2#:...,#Person2# believes in UFOs and can see them in...,UFOs and aliens
4,dev_4,#Person1#: Did you go to school today?\n#Perso...,#Person1# didn't go to school today. #Person2#...,go to school


In [8]:
test_df = pd.read_csv('/kaggle/input/dialogsum/CSV/test.csv')
print(f"test_df.shape: {test_df.shape}")
test_df.head()

test_df.shape: (1500, 4)


,id,dialogue,summary,topic
0,test_0_1,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson helps #Person1# to write a memo to ...,communication method
1,test_0_2,"#Person1#: Ms. Dawson, I need you to take a di...",In order to prevent employees from wasting tim...,company policy
2,test_0_3,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson takes a dictation for #Person1# abo...,dictation
3,test_1_1,#Person1#: You're finally here! What took so l...,#Person2# arrives late because of traffic jam....,public transportation
4,test_1_2,#Person1#: You're finally here! What took so l...,#Person2# decides to follow #Person1#'s sugges...,transportation


## Check Sample Output with 1-shot ICL

In [9]:
input_text = f"Summarize the dialogue between persons marked between double backticks. \
                \nAn example dialogue and summary is provided between triple backticks.\
                \n\n``` Example Dialogue: {test_df['dialogue'][1]}\n \
                \nExample Summary: {test_df['summary'][1]}```\
                \n\n``Dialogue: {test_df['dialogue'][0]}`` \
                \nSummary:"
input_ids = tokenizer(input_text, return_tensors="pt", padding="max_length", truncation=True).input_ids.to("cuda")

outputs = model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{input_text}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{test_df["summary"][0]}\n')
print(dash_line)
print(f'MODEL GENERATION - ONE SHOT:\n{tokenizer.decode(outputs[0])}')

---------------------------------------------------------------------------------------------------
INPUT PROMPT:
Summarize the dialogue between persons marked between double backticks.                 
An example dialogue and summary is provided between triple backticks.                

``` Example Dialogue: #Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any 

In [10]:
(train_df['dialogue'].apply(len)>512).sum()

8800

In [11]:
def mytokenizer(sample):
    start_prompt = 'Summarize the following conversation.\n\n'
    end_prompt = '\n\nSummary: '
    prompt = [start_prompt + dialogue + end_prompt for dialogue in sample["dialogue"]] # This is for batched=True
    sample['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt").input_ids
    # with tokenizer.as_target_tokenizer():
    sample['labels'] = tokenizer(sample["summary"], padding="max_length", truncation=True, return_tensors="pt").input_ids
    
    return sample

In [12]:
#'''WRONG IMPLEMENTATION'''
# def mytokenizer(sample):
#     start_prompt = 'Summarize the following conversation.\n\n'
#     end_prompt = '\n\nSummary: '
#     prompt = [start_prompt + dialogue + end_prompt for dialogue in sample["dialogue"]] # This is for batched=True
#     rt = tokenizer(prompt, padding="max_length", truncation=True)
#     with tokenizer.as_target_tokenizer():
#         rt['labels'] = tokenizer(sample["summary"], padding="max_length", truncation=True)["input_ids"]
    
#     return rt

In [13]:
train_dataset = Dataset.from_pandas(train_df)
tok_train_dataset = train_dataset.map(mytokenizer, batched=True, remove_columns=train_dataset.column_names)
tok_train_dataset

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 12460
})

In [14]:
val_dataset = Dataset.from_pandas(val_df)
tok_val_dataset = val_dataset.map(mytokenizer, batched=True, remove_columns=val_dataset.column_names)
tok_val_dataset

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 500
})

In [15]:
test_dataset = Dataset.from_pandas(test_df)
tok_test_dataset = test_dataset.map(mytokenizer, batched=True, remove_columns=test_dataset.column_names)
tok_test_dataset

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 1500
})

## PEFT LoRA

In [16]:
loraconfig = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    # inference_mode=False,
    r=8,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.01,
    bias='none',
    # use_rslora=True,
)

lora_model = get_peft_model(model, loraconfig)
# lora_model.print_trainable_parameters()
print(print_number_of_trainable_model_parameters(lora_model))

trainable model parameters: 884736
all model parameters: 248462592
percentage of trainable model parameters: 0.36%


## PEFT Model Training

In [25]:
#https://stackoverflow.com/questions/76359515/hugging-face-transformers-trainer-per-device-train-batch-size-vs-auto-find-batc
training_args = Seq2SeqTrainingArguments(
    output_dir=f"./t5-lora-finetuned-{str(int(time.time()))}",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    auto_find_batch_size=True,
    learning_rate=1e-3,
    num_train_epochs=2,
    # weight_decay=0.01,
    # save_strategy="epoch",
    # logging_dir="./logs",
    logging_steps=100,
    # fp16=True,  # If using a GPU with FP16 support
    report_to="none",  # Set to "wandb" or "tensorboard" if using logging tools
    # max_grad_norm=1.0,
    label_names=['labels'],
)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=lora_model)

trainer = Seq2SeqTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=tok_train_dataset,
    eval_dataset=tok_val_dataset,  # Optional
    # tokenizer=tokenizer,
    data_collator=data_collator,
)
trainer.train()

## Save the PEFT Model

In [30]:
peft_model_path="./peft-dialogue-summary-checkpoint-local"

trainer.model.save_pretrained(peft_model_path)

In [32]:
# !zip -r loramodel.zip /kaggle/working/peft-dialogue-summary-checkpoint-local

  adding: kaggle/working/peft-dialogue-summary-checkpoint-local/ (stored 0%)
  adding: kaggle/working/peft-dialogue-summary-checkpoint-local/README.md (deflated 66%)
  adding: kaggle/working/peft-dialogue-summary-checkpoint-local/adapter_model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 8%)
  adding: kaggle/working/peft-dialogue-summary-checkpoint-local/adapter_config.json (deflated 54%)


## Load Base Model + PEFT-LoRA Adapter for inference

In [29]:
peft_model_base = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", torch_dtype=torch.bfloat16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

peft_model = PeftModel.from_pretrained(model, 
                                       './peft-dialogue-summary-checkpoint-local', 
                                       torch_dtype=torch.bfloat16,
                                       is_trainable=False,
                                       device_map='auto')
print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 0
all model parameters: 248462592
percentage of trainable model parameters: 0.00%


## Check Sample Output after PEFT-LoRA

In [19]:
input_text = f"Summarize the dialogue between persons marked between double backticks. \
                \nAn example dialogue and summary is provided between triple backticks.\
                \n\n``` Example Dialogue: {test_df['dialogue'][1]}\n \
                \nExample Summary: {test_df['summary'][1]}```\
                \n\n``Dialogue: {test_df['dialogue'][0]}`` \
                \nSummary:"
input_ids = tokenizer(input_text, return_tensors="pt", padding="max_length", truncation=True).input_ids.to("cuda")

outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{input_text}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{test_df["summary"][0]}\n')
print(dash_line)
print(f'MODEL GENERATION - PEFT LoRA MODEL:\n{tokenizer.decode(outputs[0])}')

---------------------------------------------------------------------------------------------------
INPUT PROMPT:
Summarize the dialogue between persons marked between double backticks.                 
An example dialogue and summary is provided between triple backticks.                

``` Example Dialogue: #Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any 

## Prep the test_df

Each dialogue has 3 human baselines

In [21]:
# https://stackoverflow.com/questions/3844801/check-if-all-elements-in-a-list-are-equal
from itertools import groupby

def all_equal(iterable):
    g = groupby(iterable)
    return next(g, True) and not next(g, False)

In [22]:
test_df['parent_id'] = test_df['id'].apply(lambda x: x.split("_")[1])

assert test_df.groupby(by='parent_id').agg({'dialogue':all_equal}).all().item()==True, "Some dialogues are different for same parent_id"

modified_test_df = test_df.groupby(by='parent_id').agg({'summary':list, 'dialogue':'first'}).reset_index()
print(f"modified_test_df.shape: {modified_test_df.shape}")
modified_test_df.head()

modified_test_df.shape: (500, 3)


,parent_id,summary,dialogue
0,0,[Ms. Dawson helps #Person1# to write a memo to...,"#Person1#: Ms. Dawson, I need you to take a di..."
1,1,[#Person2# arrives late because of traffic jam...,#Person1#: You're finally here! What took so l...
2,10,[#Person2# plans to have a trip in Hebei but #...,#Person1#: Where are you going for your trip?\...
3,100,[#Person1# is crazy for Trump and voted for hi...,#Person1#: I cannot imagine if Trump were to b...
4,101,[#Person1# doesn't know how to use the ATM. #P...,#Person1#: I need to use the ATM.\n#Person2#: ...


In [37]:
dialogues = modified_test_df['dialogue']
human_baseline_summaries = modified_test_df['summary']

zeroshot_base_model_summaries = []
oneshot_base_model_summaries = []
zeroshot_peft_model_summaries = []
oneshot_peft_model_summaries = []

for idx, dialogue in tqdm(enumerate(dialogues)):
    # for 0-shot ICL & 0-shot peft
    zeroshot_prompt = f"""Summarize the following conversation marked between double backticks.

``{dialogue}``

Summary: """

    # for 1-shot ICL & 1-shot peft
    oneshot_prompt = f"""Summarize the dialogue between persons marked between double backticks.
An example dialogue and summary is provided between triple backticks.

``` Example Dialogue: {train_df['dialogue'][1]}
Example Summary: {train_df['summary'][1]}```

``Dialogue: {dialogue}`` 
Summary:"""

    
    zeroshot_input_ids = tokenizer(zeroshot_prompt, return_tensors="pt", padding="max_length", truncation=True).input_ids.to('cuda')
    oneshot_input_ids = tokenizer(oneshot_prompt, return_tensors="pt", padding="max_length", truncation=True).input_ids.to('cuda')

    # 0 shot ICL base
    zeroshot_base_model_outputs = peft_model_base.generate(input_ids=zeroshot_input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
    zeroshot_base_model_text_output = tokenizer.decode(zeroshot_base_model_outputs[0], skip_special_tokens=True)
    
    # 1 shot ICL base
    oneshot_base_model_outputs = peft_model_base.generate(input_ids=oneshot_input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
    oneshot_base_model_text_output = tokenizer.decode(oneshot_base_model_outputs[0], skip_special_tokens=True)

    # 0 shot PEFT LoRA
    zeroshot_peft_model_outputs = peft_model.generate(input_ids=zeroshot_input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
    zeroshot_peft_model_text_output = tokenizer.decode(zeroshot_peft_model_outputs[0], skip_special_tokens=True)

    # 1 shot PEFT LoRA
    oneshot_peft_model_outputs = peft_model.generate(input_ids=oneshot_input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
    oneshot_peft_model_text_output = tokenizer.decode(oneshot_peft_model_outputs[0], skip_special_tokens=True)

    zeroshot_base_model_summaries.append(zeroshot_base_model_text_output)
    oneshot_base_model_summaries.append(oneshot_base_model_text_output)
    zeroshot_peft_model_summaries.append(zeroshot_peft_model_text_output)
    oneshot_peft_model_summaries.append(oneshot_peft_model_text_output)

zipped_summaries = list(zip(dialogues, human_baseline_summaries, zeroshot_base_model_summaries, oneshot_base_model_summaries, zeroshot_peft_model_summaries, oneshot_peft_model_summaries))
 
df = pd.DataFrame(zipped_summaries, columns = ['dialogues', 'human_baseline_summaries', 'zeroshot_base_model_summaries', 'oneshot_base_model_summaries', 'zeroshot_peft_model_summaries', 'oneshot_peft_model_summaries'])
df

500it [51:23,  6.17s/it]


,dialogues,human_baseline_summaries,zeroshot_base_model_summaries,oneshot_base_model_summaries,zeroshot_peft_model_summaries,oneshot_peft_model_summaries
0,"#Person1#: Ms. Dawson, I need you to take a di...",[Ms. Dawson helps #Person1# to write a memo to...,"#Person1#: Ms. Dawson, I need you to take a di...",Summary: #Person1#: I need you to take a dicta...,Ms. Dawson asks Ms. Dawson to take a dictation...,Ms. Dawson asks Ms. Dawson to take a dictation...
1,#Person1#: You're finally here! What took so l...,[#Person2# arrives late because of traffic jam...,The driver of the car is a bit stressed.,Summary: The public transport system is good f...,#Person2# got stuck in traffic again. #Person1...,#Person2# got stuck in traffic again. #Person1...
2,#Person1#: Where are you going for your trip?\...,[#Person2# plans to have a trip in Hebei but #...,#Person1#: I'm going to Hebei for my trip. #Pe...,The north of China is experiencing severe sand...,#Person2# is going to Hebei for a trip. #Perso...,#Person2# is going to Hebei for a trip. #Perso...
3,#Person1#: I cannot imagine if Trump were to b...,[#Person1# is crazy for Trump and voted for hi...,The two speakers are discussing the future of ...,Summary: #Person1: I cannot imagine if Trump w...,#Person1# and #Person2# are discussing the fut...,#Person1# and #Person2# are discussing the fut...
4,#Person1#: I need to use the ATM.\n#Person2#: ...,[#Person1# doesn't know how to use the ATM. #P...,#Person1#: I need to use the ATM. #Person2#: I...,#Person1#: I need to use the ATM. #Person2#: I...,#Person1# needs to use the ATM. #Person2# help...,#Person1# needs to use the ATM. #Person2# help...
...,...,...,...,...,...,...
495,#Person1#: Thank you. Steven. That was the mos...,[Steven and Lin just had a great meal. Then th...,#Person1: Thank you for the wonderful meal you...,Summary: Steven and Lin are discussing tips in...,Lin thanks Steven for the wonderful meal he ha...,Lin thanks Steven for the magnificent meal he ...
496,"#Person1#: Bill, how can you hear so happy tod...",[Bill is happy because he made a move to know ...,"#Person1#: Bill, how can you hear so happy tod...",Bill moved to a new place.,Bill tells #Person1# his roommate made a move ...,Bill tells Bill his roommate made a move and h...
497,#Person1#: I'd like to pay my bill now. \n#Per...,[#Person2# checks Tom Wilson's information and...,#Person1#: I'd like to pay my bill now.,The hotel charges 660 US dollars for four nights.,Tom Wilson asks Tom to pay his bill by credit ...,Tom Wilson asks Tom Wilson to pay his bill by ...
498,#Person1#: Carol telephone.\n#Person2#: Who is...,[Susan calls Carol to ask about the party time...,Carol telephones Carrollite Susan.,Summary: Carol telephones Carrollite Susan.,Susan calls Carol to ask her if it's important...,Carol telephones Susan and asks her if the par...


In [38]:
df.to_excel('DialogSum_ROUGEscore_comparison_modifiedtestset.xlsx', index=False)

In [39]:
def myrougescorer(row, col):
    rouge_scores = []
    for ref in row['human_baseline_summaries']:
        rouge_scores.append(rouge.compute(
            predictions=[row[col]],
            references=[ref],
            use_aggregator=True,
            use_stemmer=True
        ))
    return pd.DataFrame(rouge_scores).apply(max, axis=0).to_dict()

In [40]:
rouge = evaluate.load('rouge')
tqdm.pandas()

df['zeroshot_base_model_rouge'] = df[['human_baseline_summaries','zeroshot_base_model_summaries']].progress_apply(
                                lambda x: myrougescorer(x,'zeroshot_base_model_summaries'), axis=1)
df['oneshot_base_model_rouge'] = df[['human_baseline_summaries','oneshot_base_model_summaries']].progress_apply(
                                lambda x: myrougescorer(x,'oneshot_base_model_summaries'), axis=1)
df['zeroshot_peft_model_rouge'] = df[['human_baseline_summaries','zeroshot_peft_model_summaries']].progress_apply(
                                lambda x: myrougescorer(x,'zeroshot_peft_model_summaries'), axis=1)
df['oneshot_peft_model_rouge'] = df[['human_baseline_summaries','oneshot_peft_model_summaries']].progress_apply(
                                lambda x: myrougescorer(x,'oneshot_peft_model_summaries'), axis=1)


zeroshot_base_model_results = pd.DataFrame(df['zeroshot_base_model_rouge'].values.tolist()).apply(np.mean, axis=0).to_dict()
oneshot_base_model_results = pd.DataFrame(df['oneshot_base_model_rouge'].values.tolist()).apply(np.mean, axis=0).to_dict()
zeroshot_peft_model_results = pd.DataFrame(df['zeroshot_peft_model_rouge'].values.tolist()).apply(np.mean, axis=0).to_dict()
oneshot_peft_model_results = pd.DataFrame(df['oneshot_peft_model_rouge'].values.tolist()).apply(np.mean, axis=0).to_dict()

print('BASE MODEL (0-shot ICL):')
print(zeroshot_base_model_results)
print('BASE MODEL (1-shot ICL):')
print(oneshot_base_model_results)
print('PEFT LoRA MODEL (0-shot ICL):')
print(zeroshot_peft_model_results)
print('PEFT LoRA MODEL (1-shot ICL):')
print(oneshot_peft_model_results)

100%|██████████| 500/500 [03:19<00:00,  2.51it/s]

BASE MODEL (0-shot ICL):
{'rouge1': 0.29917551504838025, 'rouge2': 0.11520027273497939, 'rougeL': 0.2579606763530079, 'rougeLsum': 0.2579606763530079}
BASE MODEL (1-shot ICL):
{'rouge1': 0.3310853163807478, 'rouge2': 0.14210090011156606, 'rougeL': 0.28074975997998886, 'rougeLsum': 0.28074975997998886}
PEFT LoRA MODEL (0-shot ICL):
{'rouge1': 0.4842449193273427, 'rouge2': 0.23612930239506774, 'rougeL': 0.3995663021036166, 'rougeLsum': 0.3995663021036166}
PEFT LoRA MODEL (1-shot ICL):
{'rouge1': 0.46796013541400194, 'rouge2': 0.22462883930272237, 'rougeL': 0.3846801801953352, 'rougeLsum': 0.3846801801953352}


In [41]:
print("Absolute percentage improvement of 1-shot PEFT MODEL over 0-shot BASE MODEL")

improvement = (np.array(list(oneshot_peft_model_results.values())) - np.array(list(zeroshot_base_model_results.values())))
for key, value in zip(oneshot_peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Absolute percentage improvement of 1-shot PEFT MODEL over 0-shot BASE MODEL
rouge1: 16.88%
rouge2: 10.94%
rougeL: 12.67%
rougeLsum: 12.67%


In [42]:
print("Absolute percentage improvement of 1-shot PEFT MODEL over 1-shot BASE MODEL")

improvement = (np.array(list(oneshot_peft_model_results.values())) - np.array(list(oneshot_base_model_results.values())))
for key, value in zip(oneshot_peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Absolute percentage improvement of 1-shot PEFT MODEL over 1-shot BASE MODEL
rouge1: 13.69%
rouge2: 8.25%
rougeL: 10.39%
rougeLsum: 10.39%


In [43]:
print("Absolute percentage improvement of 1-shot PEFT MODEL over 0-shot PEFT MODEL")

improvement = (np.array(list(oneshot_peft_model_results.values())) - np.array(list(zeroshot_peft_model_results.values())))
for key, value in zip(oneshot_peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Absolute percentage improvement of 1-shot PEFT MODEL over 0-shot PEFT MODEL
rouge1: -1.63%
rouge2: -1.15%
rougeL: -1.49%
rougeLsum: -1.49%


## Observation: 0-shot PEFT Model seems to have the best performance over others.